In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
class SafetyMLP(nn.Module):
    def __init__(self, input_size):
        super(SafetyMLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.layers(x)

In [ ]:
data_path = "../../../data/cleaned_data.csv"
df = pd.read_csv(data_path).dropna(subset=['cleaned_statement', 'label'])

X_text = df['text']
X_emotions = df[['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']]
y = df['target']

In [ ]:
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
X_tfidf = tfidf.fit_transform(X_text).toarray()
X_combined = np.hstack([X_tfidf, X_emotions.values])

X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train.values).view(-1, 1)),
    batch_size=32, shuffle=True
)

In [ ]:
model = SafetyMLP(X_train_scaled.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(50):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50, Loss: {loss.item():.4f}")

In [ ]:
os.makedirs('../local_models', exist_ok=True)
torch.save(model.state_dict(), '../local_models/safety_mlp.pth')
joblib.dump(scaler, '../local_models/scaler.pkl')
joblib.dump(tfidf, '../local_models/tfidf_vectorizer.pkl')
print("Model i artefakty zapisane w backend/ml/local_models/")